# FINAL 05 — RQ3 Misclassification, Confidence, Calibration and SHAP/LIME

**Thesis:** An Ethical Risk Assessment of Sentiment Analysis Models in Social Media Monitoring with a Focus on Bias, Inclusivity, and Misclassification Patterns  
**Student:** Abhishek Tomar | PN1196973 | LJMU | October 2026

**Purpose:** Answer RQ3 by separating complete error-direction evidence from confidence/calibration evidence and sampled post-hoc explanation evidence.

> **Final-run rule:** TweetEval test data is held out from model/feature/hyperparameter selection. External datasets are never used for source-model selection. Linguistic subgroup thresholds are frozen and are not tuned after observing subgroup sizes.

In [8]:
!pip install -q emoji vaderSentiment pandas pyarrow scikit-learn scipy google-api-python-client

from google.colab import drive, auth
drive.mount('/content/drive')

import io
import sys
from pathlib import Path
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Shared final-run utility stored in the Drive Experiments folder.
UTIL_FILE_ID = "14KfwtUf2zSd_jlW7i-6K8XgBJ62Aqk02"
UTIL_LOCAL = Path("/content/final_thesis_utils.py")

def load_final_thesis_utils():
    if not UTIL_LOCAL.exists() or UTIL_LOCAL.stat().st_size == 0:
        auth.authenticate_user()
        creds, _ = google.auth.default()
        service = build("drive", "v3", credentials=creds, cache_discovery=False)
        request = service.files().get_media(fileId=UTIL_FILE_ID)
        with io.FileIO(str(UTIL_LOCAL), "wb") as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
                if status:
                    print(f"Downloading final_thesis_utils.py: {int(status.progress()*100)}%")

    assert UTIL_LOCAL.exists() and UTIL_LOCAL.stat().st_size > 0, \
        "final_thesis_utils.py could not be downloaded."

    if "/content" not in sys.path:
        sys.path.insert(0, "/content")

    import importlib
    import final_thesis_utils as ftu
    ftu = importlib.reload(ftu)

    RUN_ROOT = Path("/content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN")
    ftu.FINAL_ROOT = RUN_ROOT
    ftu.FINAL = {
        "root": RUN_ROOT,
        "data": RUN_ROOT / "data",
        "models": RUN_ROOT / "models",
        "predictions": RUN_ROOT / "predictions",
        "results": RUN_ROOT / "results",
        "figures": RUN_ROOT / "figures",
        "manifests": RUN_ROOT / "manifests",
    }
    return ftu

ftu = load_final_thesis_utils()

globals().update({k: v for k, v in vars(ftu).items() if not k.startswith("_")})
FINAL_ROOT = ftu.FINAL_ROOT
FINAL = ftu.FINAL
setup_final_dirs()

print("✅ final_thesis_utils imported successfully")
print("✅ Final run root:", FINAL_ROOT)

!pip install -q shap lime
import json, numpy as np, pandas as pd
from lime.lime_text import LimeTextExplainer
import shap

import re

PRED_PATH = FINAL["predictions"] / "FINAL_SELECTED_TweetEval.parquet"
MODEL_PATH = FINAL["models"] / "FINAL_SELECTED_PIPELINE.pkl"

assert PRED_PATH.exists(), f"Missing FINAL_02 prediction file: {PRED_PATH}"
assert MODEL_PATH.exists(), f"Missing FINAL_02 locked model: {MODEL_PATH}"

pred = pd.read_parquet(PRED_PATH)
pipe = load_pickle(MODEL_PATH)
errors = pred[pred["correct"] == 0].copy()

assert len(pred) == 12284, f"Expected 12,284 test predictions, found {len(pred):,}"
assert len(errors) > 0, "No errors found; RQ3 cannot proceed."

print(f"✅ Loaded {len(pred):,} predictions and {len(errors):,} held-out errors.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ final_thesis_utils imported successfully
✅ Final run root: /content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN
✅ Loaded 12,284 predictions and 4,651 held-out errors.


## A. Complete error-direction analysis
This uses **all** held-out test errors, not a sample.

In [9]:
errors["Error Direction"]=errors["true_label"]+" -> "+errors["pred_label"]
overall=(errors.groupby("Error Direction").size().reset_index(name="N Errors").sort_values("N Errors",ascending=False))
overall["% of All Errors"]=100*overall["N Errors"]/len(errors)
overall.to_csv(FINAL["results"]/"F06_all_error_directions.csv",index=False)

sub=(errors.groupby(["subgroup","Error Direction"]).size().reset_index(name="N Errors"))
tot=pred.groupby("subgroup").size().rename("Subgroup N").reset_index()
sub=sub.merge(tot,on="subgroup",how="left")
sub["Rate Within Subgroup"]=sub["N Errors"]/sub["Subgroup N"]
sub.to_csv(FINAL["results"]/"F06_error_directions_by_subgroup.csv",index=False)
display(overall)

,Error Direction,N Errors,% of All Errors
0,negative -> neutral,1755,37.733821
3,neutral -> positive,1040,22.360783
5,positive -> neutral,730,15.695549
2,neutral -> negative,684,14.706515
1,negative -> positive,374,8.041281
4,positive -> negative,68,1.462051


## B. Confidence and calibration
HCER and MCE are study-defined reliability indicators. They are not external ethical standards.

In [10]:
cc=confidence_calibration_by_subgroup(pred,n_bins=10,threshold=0.70)
cc.to_csv(FINAL["results"]/"F07_confidence_calibration_by_subgroup.csv",index=False)
display(cc)

,Subgroup,N,N Errors,Misclassification Rate,HCER,MCE,ECE,Mean Confidence,Accuracy,Interpretation Status
0,emoji-heavy,710,247,0.347887,0.198381,0.585728,0.051766,0.645391,0.652113,stable_quantitative
1,other,1112,376,0.338129,0.178191,0.586130,0.037129,0.635156,0.661871,stable_quantitative
2,reference-unmarked,10388,4002,0.385252,0.145677,0.572499,0.007797,0.610885,0.614748,stable_quantitative
3,sarcasm-indicated,14,3,0.214286,0.000000,0.551196,0.179006,0.606708,0.785714,exploratory_descriptive_only
4,slang-heavy,60,23,0.383333,0.173913,0.565065,0.104879,0.627608,0.616667,stable_quantitative


## C. Stratified high-confidence explanation sample
Up to 10 highest-confidence errors are selected from each focal group and `reference-unmarked`. SHAP and LIME are treated as **post-hoc associative explanation methods**, not causal evidence.

The final descriptive coding is **multi-label**: an error instance may surface more than one cue. Cue summaries describe only the stratified explanation sample and must not be interpreted as prevalence among all errors.


In [11]:
sample_parts=[]
for sg in FOCAL_GROUPS:
    part=errors[errors.subgroup==sg].sort_values("confidence",ascending=False).head(10)
    sample_parts.append(part)
sample=pd.concat(sample_parts,ignore_index=True)
sample.to_csv(FINAL["results"]/"F08_explanation_sample_manifest.csv",index=False)
print(sample.groupby("subgroup").size())

subgroup
emoji-heavy           10
reference-unmarked    10
sarcasm-indicated      3
slang-heavy           10
dtype: int64


In [12]:
# LIME explanations
explainer=LimeTextExplainer(class_names=LABELS,random_state=SEED)
lime_rows=[]
for i,row in sample.iterrows():
    exp=explainer.explain_instance(row.text,pipe.predict_proba,num_features=12,top_labels=1)
    label=int(row.y_pred)
    feats=[x[0] for x in exp.as_list(label=label)]
    lime_rows.append({"row_id":row.row_id,"LIME Features":" | ".join(feats)})
lime_df=pd.DataFrame(lime_rows)

In [13]:
# SHAP explanation of the underlying linear SVM decision functions.
# The final classifier is CalibratedClassifierCV around LinearSVC.
# SHAP therefore explains the mean underlying LinearSVC decision function
# across calibration folds, not the post-calibration probability itself.

shap_rows = []

try:
    features = pipe.named_steps["features"]
    clf = pipe.named_steps["clf"]

    bg_text = pred.text.sample(min(200, len(pred)), random_state=SEED)
    X_bg = features.transform(bg_text)
    X_s = features.transform(sample.text)
    names = np.asarray(features.get_feature_names_out(), dtype=str)

    if hasattr(clf, "coef_"):
        coef = np.asarray(clf.coef_)
        intercept = np.asarray(clf.intercept_)
        shap_basis = "linear estimator decision function"

    elif hasattr(clf, "calibrated_classifiers_"):
        estimators = [
            cc.estimator
            for cc in clf.calibrated_classifiers_
            if hasattr(cc.estimator, "coef_")
        ]
        if not estimators:
            raise TypeError("No underlying linear estimators found inside calibrated classifier.")

        coef = np.mean(np.stack([np.asarray(e.coef_) for e in estimators]), axis=0)
        intercept = np.mean(
            np.stack([np.asarray(e.intercept_) for e in estimators]),
            axis=0,
        )
        shap_basis = "mean LinearSVC decision function across calibration folds"

    else:
        raise TypeError("Final estimator is not a supported linear/calibrated-linear model.")

    explainer = shap.LinearExplainer((coef, intercept), X_bg)
    sv = explainer(X_s)
    vals = np.asarray(sv.values)

    for j, row in sample.reset_index(drop=True).iterrows():
        arr = np.asarray(vals[j])
        label = int(row.y_pred)

        if arr.ndim == 2:
            if arr.shape[0] == len(names) and arr.shape[1] == len(LABELS):
                arr = arr[:, label]
            elif arr.shape[0] == len(LABELS) and arr.shape[1] == len(names):
                arr = arr[label, :]
            else:
                raise ValueError(f"Unexpected SHAP array shape: {arr.shape}")

        arr = np.asarray(arr).reshape(-1)
        idx = np.argsort(np.abs(arr))[-12:][::-1]

        shap_rows.append({
            "row_id": row.row_id,
            "SHAP Features": " | ".join([str(names[k]) for k in idx]),
            "SHAP Basis": shap_basis,
        })

except Exception as e:
    print("SHAP automatic extraction needs review:", repr(e))
    shap_rows = [{
        "row_id": r.row_id,
        "SHAP Features": "",
        "SHAP Basis": "",
        "SHAP Status": f"automatic extraction failed: {repr(e)}",
    } for _, r in sample.iterrows()]

shap_df = pd.DataFrame(shap_rows)


## D. Reproducible descriptive cue coding of SHAP/LIME outputs
The explanation outputs are mapped independently to the same predeclared descriptive codebook:

- **Negation/contrast cue** — explicit negation forms, contrast markers, or the model's negation-density feature.
- **Slang/informal cue** — tokens from the frozen study slang lexicon or the model's slang-ratio feature.
- **Emoji/punctuation/social-form cue** — emoji, hashtags, expressive punctuation, or corresponding social-form features.
- **Lexical polarity cue** — VADER lexicon terms with absolute valence >= 2 or the VADER-compound feature.
- **Other lexical/contextual cue** — no specific cue above is detected in the displayed top features.

These labels describe what appears in the **top explanation features**. They do not identify the cause of an error. SHAP/LIME agreement is calculated over the four specific cue families only; cases where neither method exposes a specific coded cue are left undefined (`NaN`) rather than treated as agreement.


In [14]:
import emoji
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# ------------------------------------------------------------------
# Reproducible multi-label descriptive codebook
# ------------------------------------------------------------------
SPECIFIC_CUES = [
    "Negation/contrast cue",
    "Slang/informal cue",
    "Emoji/punctuation/social-form cue",
    "Lexical polarity cue",
]
OTHER_CUE = "Other lexical/contextual cue"

NEGATION_FORMS = {
    "not","no","never","cannot","can't","won't","doesn't","don't",
    "didn't","isn't","aren't","wasn't","weren't","shouldn't",
    "wouldn't","couldn't","n't",
}
CONTRAST_MARKERS = {
    "but","however","though","although","yet","instead","despite",
    "except","whereas","while",
}
SLANG_SET = {str(x).lower() for x in SLANG_TERMS}

sid = SentimentIntensityAnalyzer()
VADER_LEXICON = {str(k).lower(): float(v) for k, v in sid.lexicon.items()}

SOCIAL_FEATURE_CUES = {
    "emoji_density": "Emoji/punctuation/social-form cue",
    "punct_intensity": "Emoji/punctuation/social-form cue",
    "hashtag_density": "Emoji/punctuation/social-form cue",
    "slang_ratio": "Slang/informal cue",
    "negation_density": "Negation/contrast cue",
    "vader_compound": "Lexical polarity cue",
}

def explanation_features(s):
    """Split the stored top-feature string into individual displayed features."""
    return [x.strip() for x in str(s).split("|") if x.strip()]

def normalise_feature_name(feature):
    """Remove FeatureUnion prefixes while preserving the actual displayed feature."""
    f = str(feature).strip().lower()
    f = re.sub(r"^(word|char|social)__", "", f)
    return f

def feature_cues(feature):
    """Map one displayed SHAP/LIME feature to zero or more descriptive cue families."""
    raw = normalise_feature_name(feature)
    cues = set()

    # Direct model social-feature names.
    for key, cue in SOCIAL_FEATURE_CUES.items():
        if key in raw:
            cues.add(cue)

    # Surface social-form evidence in displayed text/ngrams.
    if emoji.emoji_list(raw) or "#" in raw or re.search(r"[!?]", raw):
        cues.add("Emoji/punctuation/social-form cue")

    words = re.findall(r"[a-z]+(?:'[a-z]+)?", raw)

    # Frozen study slang lexicon.
    if any(w in SLANG_SET for w in words):
        cues.add("Slang/informal cue")

    # Explicit negation and a small documented contrast-marker set.
    if any(w in NEGATION_FORMS or w in CONTRAST_MARKERS for w in words):
        cues.add("Negation/contrast cue")

    # Strong polarity terms from an external lexicon already used by the model features.
    if any(abs(VADER_LEXICON.get(w, 0.0)) >= 2.0 for w in words):
        cues.add("Lexical polarity cue")

    return cues

def cue_set(feature_string):
    cues = set()
    for feat in explanation_features(feature_string):
        cues.update(feature_cues(feat))
    return cues if cues else {OTHER_CUE}

def cue_text(cues):
    return " | ".join(sorted(cues))

def specific_cue_jaccard(row):
    a = set(row["LIME Cue Set"]) & set(SPECIFIC_CUES)
    b = set(row["SHAP Cue Set"]) & set(SPECIFIC_CUES)
    union = a | b
    return (len(a & b) / len(union)) if union else np.nan

x = (
    sample
    .merge(lime_df, on="row_id", how="left")
    .merge(shap_df, on="row_id", how="left")
)

x["LIME Cue Set"] = x["LIME Features"].fillna("").map(cue_set)
x["SHAP Cue Set"] = x["SHAP Features"].fillna("").map(cue_set)
x["LIME Cues"] = x["LIME Cue Set"].map(cue_text)
x["SHAP Cues"] = x["SHAP Cue Set"].map(cue_text)
x["Shared Specific Cues"] = x.apply(
    lambda r: cue_text(
        (set(r["LIME Cue Set"]) & set(r["SHAP Cue Set"])) & set(SPECIFIC_CUES)
    ),
    axis=1,
)
x["SHAP-LIME Cue Jaccard"] = x.apply(specific_cue_jaccard, axis=1)

# Do not persist Python set objects.
instance_out = x.drop(columns=["LIME Cue Set", "SHAP Cue Set"]).copy()
instance_out.to_csv(
    FINAL["results"] / "F08_SHAP_LIME_error_instances.csv",
    index=False,
)

# ------------------------------------------------------------------
# Method-specific cue summary.
# Percentages are NOT mutually exclusive because coding is multi-label.
# ------------------------------------------------------------------
summary_rows = []
for method in ["LIME", "SHAP"]:
    set_col = f"{method} Cue Set"

    for cue in SPECIFIC_CUES:
        mask = x[set_col].map(lambda s: cue in s)
        summary_rows.append({
            "Method": method,
            "Cue": cue,
            "N Sampled Cases": int(mask.sum()),
            "% of Explanation Sample": 100.0 * float(mask.mean()),
            "Interpretation": "descriptive multi-label cue presence in top explanation features",
        })

    other_mask = x[set_col].map(lambda s: OTHER_CUE in s)
    summary_rows.append({
        "Method": method,
        "Cue": OTHER_CUE,
        "N Sampled Cases": int(other_mask.sum()),
        "% of Explanation Sample": 100.0 * float(other_mask.mean()),
        "Interpretation": "no specific predeclared cue detected in displayed top features",
    })

cue_summary = pd.DataFrame(summary_rows)
cue_summary.to_csv(
    FINAL["results"] / "F08_explanation_cue_summary.csv",
    index=False,
)

# Persist the operational codebook for thesis traceability.
codebook = pd.DataFrame([
    {
        "Cue": "Negation/contrast cue",
        "Operational Rule": "explicit negation forms, fixed contrast markers, or negation_density feature",
        "Claim Boundary": "descriptive cue in top explanation features; not causal error mechanism",
    },
    {
        "Cue": "Slang/informal cue",
        "Operational Rule": "frozen study slang lexicon or slang_ratio feature",
        "Claim Boundary": "descriptive cue in top explanation features; not causal error mechanism",
    },
    {
        "Cue": "Emoji/punctuation/social-form cue",
        "Operational Rule": "emoji, hashtag, !/?, emoji_density, punct_intensity, or hashtag_density feature",
        "Claim Boundary": "descriptive cue in top explanation features; not causal error mechanism",
    },
    {
        "Cue": "Lexical polarity cue",
        "Operational Rule": "VADER term with |valence| >= 2 or vader_compound feature",
        "Claim Boundary": "descriptive cue in top explanation features; not causal error mechanism",
    },
    {
        "Cue": "Other lexical/contextual cue",
        "Operational Rule": "none of the four specific predeclared cues detected",
        "Claim Boundary": "residual descriptive category only",
    },
])
codebook.to_csv(
    FINAL["results"] / "F08_explanation_coding_codebook.csv",
    index=False,
)

# Remove the superseded forced single-category taxonomy if it exists.
legacy_taxonomy = FINAL["results"] / "F08_explanation_supported_taxonomy.csv"
if legacy_taxonomy.exists():
    legacy_taxonomy.unlink()

agreement = pd.DataFrame([{
    "N Explanation Cases": len(x),
    "N With Defined Specific-Cue Agreement": int(x["SHAP-LIME Cue Jaccard"].notna().sum()),
    "Mean SHAP-LIME Cue Jaccard": float(x["SHAP-LIME Cue Jaccard"].mean())
        if x["SHAP-LIME Cue Jaccard"].notna().any() else np.nan,
    "Median SHAP-LIME Cue Jaccard": float(x["SHAP-LIME Cue Jaccard"].median())
        if x["SHAP-LIME Cue Jaccard"].notna().any() else np.nan,
}]
)
agreement.to_csv(
    FINAL["results"] / "F08_SHAP_LIME_cue_agreement.csv",
    index=False,
)

display(cue_summary)
display(agreement)
print("✅ Reproducible multi-label explanation coding complete.")
print("✅ Superseded forced single-category taxonomy removed.")


,Method,Cue,N Sampled Cases,% of Explanation Sample,Interpretation
0,LIME,Negation/contrast cue,5,15.151515,descriptive multi-label cue presence in top ex...
1,LIME,Slang/informal cue,10,30.303030,descriptive multi-label cue presence in top ex...
2,LIME,Emoji/punctuation/social-form cue,0,0.000000,descriptive multi-label cue presence in top ex...
3,LIME,Lexical polarity cue,16,48.484848,descriptive multi-label cue presence in top ex...
4,LIME,Other lexical/contextual cue,9,27.272727,no specific predeclared cue detected in displa...
5,SHAP,Negation/contrast cue,7,21.212121,descriptive multi-label cue presence in top ex...
6,SHAP,Slang/informal cue,9,27.272727,descriptive multi-label cue presence in top ex...
7,SHAP,Emoji/punctuation/social-form cue,16,48.484848,descriptive multi-label cue presence in top ex...
8,SHAP,Lexical polarity cue,25,75.757576,descriptive multi-label cue presence in top ex...
9,SHAP,Other lexical/contextual cue,2,6.060606,no specific predeclared cue detected in displa...


,N Explanation Cases,N With Defined Specific-Cue Agreement,Mean SHAP-LIME Cue Jaccard,Median SHAP-LIME Cue Jaccard
0,33,31,0.526882,0.5


✅ Reproducible multi-label explanation coding complete.
✅ Superseded forced single-category taxonomy removed.
